# Exam Preparation: SAXS and Least-Squares Fitting

In this notebook you will cover two main concepts:

1. **Small-angle scattering of dilute spheres** — including $R_g$ and the Guinier region
2. **Manual least-squares fitting** — searching a parameter space to match an unknown distribution

For the sections marked **Your Task**, you must write your own code. You may use AI assistance, but you should understand every line before the exam. **You should be able to explain everything in this notebook on the exam.** Answers will be posted on Friday.  

**AI Disclosure**: I prepared this entire notebook with examples by hand; after it was finished, I ran it through Claude to check for errors and clean it up. It added a bit to the markup, revised the numbering/organization, and laid out the "your tasks" a bit more cleanly along with the pencil emoji.

## Setup

Run these two cells first to import libraries and download the course data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
import pandas as pd
from scipy.stats import linregress

In [ ]:
# Download course data repository (only needs to run once per session)
!git clone https://github.com/cbishop4/MSE7530.git

---
# Part 1: Least-Squares Fitting of Scattering from a Bidisperse Sphere Solution

In this exercise you will use the analytical form factor for dilute spheres to fit an unknown bidisperse distribution.

## 1.1 Form factor for a dilute sphere

The squared form factor (equal to the scattered intensity) for a dilute solution of spheres of radius $r$ is:

$$P(Q, r) = \left( \frac{3\,[\sin(Qr) - Qr\cos(Qr)]}{Q^3 r^3} \right)^2$$

The function below evaluates this expression.

In [ ]:
def FQ2_sphere(radius, q):
    """Squared form factor for a dilute sphere of given radius at wavevector q."""
    return (3 * (np.sin(q * radius) - q * radius * np.cos(q * radius)) /
            (q**3 * radius**3))**2

## 1.2 Monodisperse scattering

We can visualise the scattering from a single population of 100 nm spheres.
Note the fringe spacing — **it is important that you understand how fringe spacing relates to sphere size.**

In [ ]:
qrange = np.logspace(-3, 0, 1000)  # Q values evenly spaced on a log scale (nm⁻¹)
scatter = FQ2_sphere(100., qrange)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(np.log10(qrange), np.log10(scatter))
axes[0].set_xlabel(r'$\log_{10}(Q)$')
axes[0].set_ylabel(r'$\log_{10}$(Intensity)')
axes[0].set_title('Log–log scale')

axes[1].plot(qrange, np.log10(scatter))
axes[1].set_xlabel(r'$Q\ (\mathrm{nm}^{-1})$')
axes[1].set_ylabel(r'$\log_{10}$(Intensity)')
axes[1].set_title('Linear Q, log Intensity')

fig.suptitle('Simulated scattering from dilute 100 nm spheres')
plt.tight_layout()

## 1.3 Guinier region check

In the Guinier region, for a **sphere** specifically, the intensity approximates to:

$$I(Q) \approx \Delta\rho^2 V_p^2 \, e^{-Q^2 R^2/5}$$

Taking the natural log: $\ln I = \text{const} - \frac{R^2}{5}Q^2$.

A plot of $\ln I$ vs $Q^2$ (a **Guinier plot**) should therefore be linear at low $Q$, with slope $= -R^2/5$.

> **Note:** For an arbitrary particle shape, the exponent is $-R_g^2/3$ (we use this in Part 2).
> For a solid sphere, $R_g^2 = 3R^2/5$, which gives the factor of $1/5$ above.

In [ ]:
q2  = qrange**2
lnI = np.log(scatter)

fig, ax = plt.subplots()
ax.plot(q2, lnI)
ax.set_xlabel(r'$Q^2\ (\mathrm{nm}^{-2})$')
ax.set_ylabel(r'$\ln(I)$')
ax.set_title('Guinier plot — 100 nm spheres')
ax.set_xlim(0, 0.003)

In [ ]:
# Linear fit in the Guinier regime (Q² < 0.001 nm⁻²)
cut_ind  = np.where(q2 < 0.001)
cut_q2   = q2[cut_ind]
cut_lnI  = lnI[cut_ind]

slope, intercept, r_value, p_value, std_err = linregress(cut_q2, cut_lnI)

fig, ax = plt.subplots()
ax.plot(q2, lnI, label='Simulated scattering')
ax.plot(cut_q2, slope * cut_q2 + intercept, label='Linear fit', linestyle='--')
ax.set_xlabel(r'$Q^2\ (\mathrm{nm}^{-2})$')
ax.set_ylabel(r'$\ln(I)$')
ax.set_title('Guinier plot — 100 nm spheres')
ax.set_xlim(0, 0.002)
ax.legend()
ax.text(0.0005, -20, f'Slope = {slope:.2f}\nIntercept = {intercept:.2f}')

In [ ]:
# Extract radius from Guinier slope
R_guinier = (-slope * 5)**0.5
print(f'Input radius:          100.0 nm')
print(f'Guinier-extracted R:   {R_guinier:.2f} nm')
print()
print('The small discrepancy comes from the choice of Q² cutoff.')
print('The fit is only truly linear in the limit Q → 0.')

## 1.4 Bidisperse scattering

The intensity from a **bidisperse** solution (two sphere populations) is a weighted sum:

$$I(Q) = x_1 \, P(Q, r_1) + (1 - x_1) \, P(Q, r_2)$$

where $x_1$ is the number fraction of spheres with radius $r_1$.

The plot below shows how $x_1$ affects the scattering for $r_1 = 30$ nm and $r_2 = 40$ nm.

In [ ]:
r1, r2 = 30., 40.
proportions = np.linspace(0., 1., 11)
pcol = iter(cm.viridis(np.linspace(0, 1, len(proportions))))

fig, ax = plt.subplots()
for p in proportions:
    scatter_bi = FQ2_sphere(r1, qrange) * p + FQ2_sphere(r2, qrange) * (1 - p)
    ax.plot(qrange, scatter_bi, color=next(pcol), label=f'{round(p, 1)}')

ax.set_xlabel(r'$Q\ (\mathrm{nm}^{-1})$')
ax.set_ylabel('Intensity (a.u.)')
ax.set_title('Bidisperse sphere scattering')
ax.legend(title=r'$x_1$  ($r_1 = 30$ nm)', bbox_to_anchor=(1, 1))
ax.loglog()
ax.set_xlim(0.01, 1.1)
plt.tight_layout()

## 1.5 Load the unknown distribution

Below we load the target scattering data. Your goal is to find $r_1$, $r_2$, and $x_1$ that best reproduce it.

In [ ]:
target = pd.read_csv('MSE7530/sampledata/unknown_distribution.csv')

fig, ax = plt.subplots()
ax.plot(target['q (nm-1)'], target['I'])
ax.loglog()
ax.set_xlabel(r'$Q\ (\mathrm{nm}^{-1})$')
ax.set_ylabel('Intensity (a.u.)')
ax.set_title('Unknown distribution — target data')

In [ ]:
# Convert to log₁₀ intensity and numpy arrays for fitting
# Using log₁₀(I) gives more equal weight to features at low and high Q
log_uk_scatter = np.asarray(np.log10(target['I']))
q_uk           = np.asarray(target['q (nm-1)'])

## 1.6 Example: RSS for a single parameter set

The **residual sum of squares (RSS)** measures how well a simulated curve matches the data:

$$\text{RSS} = \sqrt{\sum_i \left[\log_{10}(I_\text{data},_i) - \log_{10}(I_\text{sim},_i)\right]^2}$$

Here is the RSS for one example pair ($r_1 = 30$ nm, $r_2 = 40$ nm, $x_1 = 0.5$).

In [ ]:
r1, r2, p = 30., 40., 0.5
sim_scatter     = FQ2_sphere(r1, q_uk) * p + FQ2_sphere(r2, q_uk) * (1 - p)
log_sim_scatter = np.log10(sim_scatter)
RSS             = np.sqrt(np.sum((log_uk_scatter - log_sim_scatter)**2))

fig, ax = plt.subplots()
ax.plot(q_uk, log_uk_scatter,     label='Target data')
ax.plot(q_uk, log_sim_scatter, '--', label=f'Trial fit (r₁=30, r₂=40, x₁=0.5)')
ax.set_xlabel(r'$Q\ (\mathrm{nm}^{-1})$')
ax.set_ylabel(r'$\log_{{10}}(I)$')
ax.set_title('Example trial fit')
ax.semilogx()
ax.legend()
print(f'RSS = {RSS:.4f}')

## 1.7 Scanning over one radius pair

Here we show that scanning $x_1$ for a fixed $(r_1, r_2)$ finds a minimum RSS — but we need to search all radius combinations too.

In [ ]:
test_props = np.linspace(0., 1., 11)
test_r     = np.linspace(10., 200., 20)
print(f'Grid size: {len(test_props)} × {len(test_r)} × {len(test_r)} = '
      f'{len(test_props) * len(test_r)**2} parameter sets')

In [ ]:
# Example: scan x₁ for r₁=30 nm, r₂=40 nm
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pcol = iter(cm.rainbow(np.linspace(0, 1, len(test_props))))
rsums = np.zeros(len(test_props))
r1, r2 = 30., 40.

axes[0].plot(q_uk, log_uk_scatter, 'k', label='Target', linewidth=1.5)
for i, p in enumerate(test_props):
    sim  = FQ2_sphere(r1, q_uk) * p + FQ2_sphere(r2, q_uk) * (1 - p)
    lsim = np.log10(sim)
    rsums[i] = np.sqrt(np.sum((log_uk_scatter - lsim)**2))
    axes[0].plot(q_uk, lsim, color=next(pcol), label=f'{round(p,1)}', alpha=0.7)

axes[0].semilogx()
axes[0].set_xlabel(r'$Q\ (\mathrm{nm}^{-1})$')
axes[0].set_ylabel(r'$\log_{{10}}(I)$')
axes[0].legend(title='$x_1$', fontsize=7, ncol=2)
axes[0].set_title('Trial fits: r₁=30 nm, r₂=40 nm')

axes[1].plot(test_props, rsums, 'o-')
axes[1].set_xlabel('$x_1$ (fraction of 30 nm spheres)')
axes[1].set_ylabel('RSS')
axes[1].set_title('RSS vs $x_1$ for 30 & 40 nm pair')
plt.tight_layout()

---
## ✏️ Your Task — Part 1

**Find the values of $r_1$, $r_2$, and $x_1$ that give the best fit to the unknown distribution.**

You have already seen how to compute RSS for one parameter set and scan over $x_1$ for a fixed radius pair. Now extend this to search the full 3-D parameter grid (`test_r` × `test_r` × `test_props`).

**Deliverables:**
1. Code that searches the parameter grid and records the RSS for every combination.
2. The best-fit values of $r_1$, $r_2$, and $x_1$.
3. A plot comparing your best-fit simulated curve to the target data.

**Bonus / exam connection:** Once you have your answer, go back to the lecture slides on optimization and identify how each variable in your code connects to the formal mathematical definitions of least-squares fitting.

> *Hint: you can store results in a 3-D numpy array (`rss_grid[i, j, k]`) indexed by `(r1_idx, r2_idx, prop_idx)` and use `np.argmin` to find the minimum.*

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here


---
# Part 2: Guinier Fitting of Protein $R_g$

For the **ABC transporter ATP-binding protein** ([SASDW27](https://www.sasbdb.org/data/SASDW27/)), determine $R_g$ by Guinier analysis.

For any sufficiently dilute particle — regardless of shape — the Guinier approximation gives:

$$I(Q) \approx \Delta\rho^2 V_p^2 \, e^{-Q^2 R_g^2/3}$$

Therefore a plot of $\ln I$ vs $Q^2$ is linear at low $Q$, with slope $= -R_g^2/3$, so:

$$R_g = \sqrt{-3 \times \text{slope}}$$

The data are loaded below.

In [ ]:
# Load SAXS data for the ABC transporter protein
# (unusual format — column renaming required)
ABC_data = pd.read_csv('MSE7530/sampledata/HW4/SASDW27.dat', sep=' ', header=3)
ABC_data = ABC_data.rename(columns={
    ABC_data.columns[1]: 'q',
    ABC_data.columns[3]: 'I',
    ABC_data.columns[5]: 'sigma',
})
ABC_data = ABC_data.drop(['Unnamed: 0', 'Unnamed: 2', 'Unnamed: 4', 'sigma'], axis=1)
ABC_data.head(10)

In [ ]:
# Preview the full dataset
fig, ax = plt.subplots()
ax.plot(ABC_data['q'], ABC_data['I'])
ax.set_xlabel(r'$Q\ (\mathrm{nm}^{-1})$')
ax.set_ylabel('Intensity (a.u.)')
ax.set_title('SAXS data — ABC transporter ATP-binding protein (SASDW27)')
ax.loglog()

---
## ✏️ Your Task — Part 2

**Find the $R_g$ of the ABC transporter protein using a Guinier plot.**

Follow the same approach demonstrated in Section 1.3:
1. Construct a Guinier plot ($\ln I$ vs $Q^2$).
2. Identify a suitable linear region at low $Q$ (consider $QR_g < 1.3$ as a rough guide for the upper limit).
3. Fit a line to that region and extract $R_g$ from the slope.
4. Show your Guinier plot with the linear fit overlaid.

> *Think about: how sensitive is your answer to the choice of $Q$ cutoff? What does that tell you about the reliability of the Guinier approximation?*

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here
